#### Environment Setup & Raw Data Ingestion

In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

print("Loading datasets...")
# Make sure these paths are perfectly correct for your computer
all_data_file = r"D:\Courses\IIT\AIDS\FYP\Preventing-Mechanism\PreProcessing\MBP ControllerData 0521760 Overlock.xlsx"
breakdown_file = r"D:\Courses\IIT\AIDS\FYP\Preventing-Mechanism\PreProcessing\Synthetic Overlock Breakdowns.xlsx"

df_all = pd.read_excel(all_data_file)
df_breakdown = pd.read_excel(breakdown_file)

# Handle column spelling differences
col_healthy = 'Breakdown' if 'Breakdown' in df_all.columns else 'Brekadown'
col_broken = 'Breakdown' if 'Breakdown' in df_breakdown.columns else 'Brekadown'

# Isolate the healthy data 
df_healthy = df_all[df_all[col_healthy].isna()].copy()
print(f"Loaded {len(df_healthy)} healthy records and {len(df_breakdown)} breakdown records.")

ImportError: Could not find the DLL(s) 'msvcp140_1.dll'. TensorFlow requires that these DLLs be installed in a directory that is named in your %PATH% environment variable. You may install these DLLs by downloading "Microsoft C++ Redistributable for Visual Studio 2015, 2017 and 2019" for your platform from this URL: https://support.microsoft.com/help/2977003/the-latest-supported-visual-c-downloads

#### Feature Engineering & Data Fusion

In [ ]:
print("Combining Vibration, Electrical, and Mechanical Data...")

def parse_vibration_to_bands(vib_str):
    if pd.isna(vib_str): return {}
    parts = str(vib_str).split(',')
    res = {}
    try:
        for i in range(0, len(parts)-1, 2):
            f_start = int(parts[i])
            f_end = f_start + 10
            res[f"{f_start}-{f_end}Hz"] = int(parts[i+1])
    except Exception:
        pass
    return res

# The 10 extra features to track alongside vibration
extra_features = [ 
    'machineVoltageMean', 'machineVoltageMin', 'machineVoltageMax',
    'machineCurrentMean', 'machineCurrentMin', 'machineCurrentMax',
    'machinePowerMean', 'machinePowerMin', 'machinePowerMax'
]

# Process Healthy Data
h_vibs = pd.DataFrame(df_healthy['machineVibration'].apply(parse_vibration_to_bands).tolist()).fillna(0)
h_extras = df_healthy[extra_features].reset_index(drop=True)
df_h_ml = pd.concat([h_extras, h_vibs], axis=1)
df_h_ml['Label'] = 'Healthy Baseline'

# Process Breakdown Data
b_vibs = pd.DataFrame(df_breakdown['machineVibration'].apply(parse_vibration_to_bands).tolist()).fillna(0)
b_extras = df_breakdown[extra_features].reset_index(drop=True)
df_b_ml = pd.concat([b_extras, b_vibs], axis=1)
df_b_ml['Label'] = df_breakdown[col_broken].values

# Combine into one Master Dataset
df_ml_master = pd.concat([df_h_ml, df_b_ml]).fillna(0)
print(f"Master Dataset created! The AI will look at {df_ml_master.shape[1] - 1} different variables per machine.")

In [ ]:
df_ml_master

#### Label Encoding & Feature Standardization

In [ ]:
print("Formatting Data for Deep Learning...")

# Separate numbers (X) from labels (Y)
X_raw = df_ml_master.drop(columns=['Label']).values
y_text = df_ml_master['Label'].values

# Encode text labels into numbers
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_text)
y_categorical = to_categorical(y_encoded)

# Scale the data so RPM (4000) and Current (1.5) are treated equally
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Save the Scaler and Encoder to your hard drive
with open('sewing_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('sewing_encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)
    
print("Translators saved successfully.")

#### Time-Series Transformation (Sliding Window Generation)

In [ ]:
print("Creating Sliding Time Windows for Predictive Maintenance...")

def create_sequences(data_X, data_Y, time_steps=5):
    X_seq, y_seq = [], []
    for i in range(len(data_X) - time_steps):
        # Grab a block of past data (e.g., 5 consecutive readings)
        X_seq.append(data_X[i : (i + time_steps)])
        # Grab the target label from the FUTURE
        y_seq.append(data_Y[i + time_steps]) 
    return np.array(X_seq), np.array(y_seq)

# Set how many past readings the AI should look at (Window Size)
TIME_STEPS = 5 
X_lstm, y_lstm = create_sequences(X_scaled, y_categorical, time_steps=TIME_STEPS)

# Split data: shuffle=False is CRITICAL to keep the timeline chronological!
X_train, X_test, y_train, y_test = train_test_split(X_lstm, y_lstm, test_size=0.2, random_state=42, shuffle=False)

print(f"Corrected LSTM Input Shape: {X_train.shape}")
print(f"The AI will look at {X_train.shape[1]} past records to predict the future.")

#### Deep Learning Architecture Definition

In [ ]:
print("Building the Predictive LSTM Brain...")

model = Sequential()

# First Layer: Reads the sliding time window
model.add(LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dropout(0.2))

# Second Layer: Deep pattern extraction
model.add(LSTM(32, return_sequences=False))
model.add(Dropout(0.2))

# Dense processing
model.add(Dense(32, activation='relu'))

# Output Layer: Predicts the exact breakdown category
num_classes = y_categorical.shape[1]
model.add(Dense(num_classes, activation='softmax'))

model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

#### Model Training, Evaluation, & Export

In [ ]:
print("Commencing Model Training...")

history = model.fit(
    X_train, y_train,
    epochs=50,           
    batch_size=16,       
    validation_data=(X_test, y_test), 
    verbose=1
)

loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print("\n" + "="*40)
print(f"FINAL PREDICTIVE TEST ACCURACY: {accuracy * 100:.2f}%")
print("="*40)

model.save("sewing_machine_predictive_lstm.h5")
print("✅ SUCCESS! Saved the .h5 model to your computer.")